# Entrenamiento y comparación de estrategias predictivas

Compara un baseline trivial, regresión logística, Random Forest, ExtraTrees e HistGradientBoosting para predecir `falla_30d`.

**Validación:** holdout temporal (últimos 6 cortes), GroupKFold por activo, LeaveOneGroupOut por empresa y TimeSeriesSplit por fecha. La selección del modelo se hace con GroupKFold por activo; el holdout temporal se usa como reporte final.

**Métricas:** precision, recall, F1, ROC-AUC y PR-AUC (más matriz de confusión). No se usa accuracy porque el objetivo está desbalanceado.

In [1]:

from pathlib import Path
import json
import warnings
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold,
    LeaveOneGroupOut,
    TimeSeriesSplit,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATASET = ROOT / "data" / "processed" / "predictive_maintenance_dataset.csv"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
MODEL_DIR = ROOT / "models"
for directory in (REPORTS, FIGURES, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

TARGET = "falla_30d"
CATEGORICAL = [
    "tipo_activo",
    "sede",
    "criticidad",
    "tipo_ultimo_mantenimiento",
    "causa_ultima_falla",
]
NUMERIC = [
    "antiguedad_dias",
    "fallas_90d",
    "fallas_180d",
    "dias_desde_ultimo_mantenimiento",
    "dias_desde_ultima_falla",
    "mtbf_dias",
    "prioridad_promedio",
    "resolucion_promedio_h",
]
FEATURES = CATEGORICAL + NUMERIC

df = pd.read_csv(DATASET, parse_dates=["fecha_corte", "ventana_inicio", "ventana_fin"])
df = df.sort_values(["fecha_corte", "tenant_id", "asset_id"]).reset_index(drop=True)
df["asset_group"] = df["tenant_id"].astype(str) + "|" + df["asset_id"].astype(str)

assert set(FEATURES).issubset(df.columns)
assert set([TARGET]).issubset(df.columns)

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
asset_groups = df["asset_group"]
tenant_groups = df["tenant_id"]

print(f"Filas: {len(df):,} | variables: {len(FEATURES)} | tasa positiva: {y.mean():.2%}")
print(df.groupby("tenant_id")["falla_30d"].agg(["size", "mean"]).to_string())


Filas: 12,000 | variables: 13 | tasa positiva: 24.49%
                    size      mean
tenant_id                         
acero-norte         3000  0.180667
alimentos-valle     3000  0.272333
logistica-pacifico  3000  0.333000
textiles-andinos    3000  0.193667


## 1. Preprocesamiento y modelos

Los pipelines se ajustan dentro de cada fold; el preprocesamiento nunca ve el conjunto de prueba.

In [2]:

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC),
        ("categorical", categorical_pipeline, CATEGORICAL),
    ],
    remainder="drop",
)

MODEL_REGISTRY = {
    "Dummy (prior)": DummyClassifier(strategy="prior"),
    "LogisticRegression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.06,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        class_weight="balanced",
        random_state=42,
    ),
}

print("Modelos configurados: " + ", ".join(MODEL_REGISTRY))


Modelos configurados: Dummy (prior), LogisticRegression, RandomForest, ExtraTrees, HistGradientBoosting


## 2. Evaluación de las cuatro estrategias de validación

In [3]:

def build_pipeline(model_name: str) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("model", clone(MODEL_REGISTRY[model_name])),
        ]
    )


def scores(y_true: pd.Series, probabilities: np.ndarray) -> dict:
    predictions = (probabilities >= 0.5).astype(int)
    result = {
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "pr_auc": average_precision_score(y_true, probabilities),
    }
    try:
        result["roc_auc"] = roc_auc_score(y_true, probabilities)
    except ValueError:
        result["roc_auc"] = np.nan
    return result


def evaluate_holdout(model_name: str) -> list[dict]:
    cutoff_values = np.sort(df["fecha_corte"].unique())
    test_cutoffs = cutoff_values[-6:]
    train_mask = ~df["fecha_corte"].isin(test_cutoffs)
    test_mask = df["fecha_corte"].isin(test_cutoffs)

    pipeline = build_pipeline(model_name)
    pipeline.fit(X.loc[train_mask], y.loc[train_mask])
    probabilities = pipeline.predict_proba(X.loc[test_mask])[:, 1]
    metrics = scores(y.loc[test_mask], probabilities)

    return [
        {
            "model": model_name,
            "strategy": "holdout_temporal",
            "fold": 0,
            "n_train": int(train_mask.sum()),
            "n_test": int(test_mask.sum()),
            **metrics,
        }
    ]


def evaluate_cv(model_name: str, splitter, strategy: str, groups=None) -> list[dict]:
    rows = []
    split_args = (X, y, groups) if groups is not None else (X, y)

    for fold, (train_index, test_index) in enumerate(splitter.split(*split_args)):
        pipeline = build_pipeline(model_name)
        pipeline.fit(X.iloc[train_index], y.iloc[train_index])
        probabilities = pipeline.predict_proba(X.iloc[test_index])[:, 1]
        metrics = scores(y.iloc[test_index], probabilities)

        rows.append(
            {
                "model": model_name,
                "strategy": strategy,
                "fold": fold,
                "n_train": int(len(train_index)),
                "n_test": int(len(test_index)),
                **metrics,
            }
        )

    return rows


results = []

for model_name in MODEL_REGISTRY:
    results.extend(evaluate_holdout(model_name))
    results.extend(
        evaluate_cv(
            model_name,
            GroupKFold(n_splits=5),
            "group_kfold_asset",
            asset_groups,
        )
    )
    results.extend(
        evaluate_cv(
            model_name,
            LeaveOneGroupOut(),
            "leave_one_tenant_out",
            tenant_groups,
        )
    )
    results.extend(
        evaluate_cv(
            model_name,
            TimeSeriesSplit(n_splits=5),
            "time_series_cv",
        )
    )

results_df = pd.DataFrame(results)

print(f"Evaluaciones: {len(results_df)}")
print(
    results_df.groupby(["model", "strategy"])["pr_auc"]
    .mean()
    .sort_values(ascending=False)
    .to_string()
)


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Evaluaciones: 75
model                 strategy            
LogisticRegression    holdout_temporal        0.460808
RandomForest          holdout_temporal        0.435433
ExtraTrees            holdout_temporal        0.424141
LogisticRegression    time_series_cv          0.409623
HistGradientBoosting  holdout_temporal        0.405923
LogisticRegression    group_kfold_asset       0.396878
RandomForest          time_series_cv          0.386305
ExtraTrees            time_series_cv          0.377717
RandomForest          group_kfold_asset       0.377341
LogisticRegression    leave_one_tenant_out    0.372195
RandomForest          leave_one_tenant_out    0.351621
ExtraTrees            group_kfold_asset       0.350034
HistGradientBoosting  time_series_cv          0.348389
ExtraTrees            leave_one_tenant_out    0.345298
HistGradientBoosting  group_kfold_asset       0.341289
                      leave_one_tenant_out    0.327287
Dummy (prior)         holdout_temporal        0.284000
     

## 3. Comparación agregada

PR-AUC es la métrica principal; F1 y recall permiten leer el costo operativo del umbral 0.5.

In [4]:

metric_columns = ["precision", "recall", "f1", "roc_auc", "pr_auc"]

comparison = (
    results_df.groupby(["model", "strategy"])[metric_columns]
    .mean()
    .reset_index()
)
fold_counts = (
    results_df.groupby(["model", "strategy"])["fold"]
    .nunique()
    .rename("folds")
    .reset_index()
)
comparison = comparison.merge(fold_counts, on=["model", "strategy"])
comparison = comparison.sort_values(["strategy", "pr_auc"], ascending=[True, False])

results_df.to_csv(REPORTS / "model_comparison_folds.csv", index=False)
comparison.to_csv(REPORTS / "model_comparison.csv", index=False)

print(comparison.round(4).to_string(index=False))

pr_pivot = comparison.pivot(index="model", columns="strategy", values="pr_auc")
pr_pivot = pr_pivot.loc[pr_pivot.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(12, 5))
pr_pivot.plot(kind="bar", ax=ax, colormap="viridis")
ax.set_title("PR-AUC por modelo y estrategia de validación")
ax.set_ylabel("PR-AUC")
ax.set_xlabel("Modelo")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Estrategia", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "model_comparison_pr_auc.png", dpi=150, bbox_inches="tight")
plt.close(fig)


               model             strategy  precision  recall     f1  roc_auc  pr_auc  folds
  LogisticRegression    group_kfold_asset     0.3580  0.6192 0.4535   0.6665  0.3969      5
        RandomForest    group_kfold_asset     0.4026  0.3212 0.3565   0.6474  0.3773      5
          ExtraTrees    group_kfold_asset     0.3387  0.4963 0.4025   0.6249  0.3500      5
HistGradientBoosting    group_kfold_asset     0.3339  0.5112 0.4036   0.6208  0.3413      5
       Dummy (prior)    group_kfold_asset     0.0000  0.0000 0.0000   0.5000  0.2449      5
  LogisticRegression     holdout_temporal     0.4061  0.6749 0.5071   0.6785  0.4608      1
        RandomForest     holdout_temporal     0.4660  0.3697 0.4123   0.6591  0.4354      1
          ExtraTrees     holdout_temporal     0.4110  0.4906 0.4473   0.6444  0.4241      1
HistGradientBoosting     holdout_temporal     0.3893  0.5364 0.4511   0.6354  0.4059      1
       Dummy (prior)     holdout_temporal     0.0000  0.0000 0.0000   0.5000  0.

## 4. Mejor modelo, matriz de confusión y umbral

In [5]:

group_metrics = comparison[comparison["strategy"].eq("group_kfold_asset")]
best_model_name = (
    group_metrics.sort_values(["pr_auc", "f1"], ascending=False).iloc[0]["model"]
)
best_holdout = results_df[
    (results_df["model"] == best_model_name)
    & (results_df["strategy"] == "holdout_temporal")
].iloc[0]
best_group = group_metrics[group_metrics["model"] == best_model_name].iloc[0]

cutoff_values = np.sort(df["fecha_corte"].unique())
test_cutoffs = cutoff_values[-6:]
train_mask = ~df["fecha_corte"].isin(test_cutoffs)
test_mask = df["fecha_corte"].isin(test_cutoffs)

best_pipeline = build_pipeline(best_model_name)
best_pipeline.fit(X.loc[train_mask], y.loc[train_mask])
best_probabilities = best_pipeline.predict_proba(X.loc[test_mask])[:, 1]
best_predictions = (best_probabilities >= 0.5).astype(int)

cm = confusion_matrix(y.loc[test_mask], best_predictions)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Sin falla", "Falla 30d"],
)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title(f"Matriz de confusión holdout: {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURES / "confusion_matrix_best_model.png", dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Mejor modelo por GroupKFold de activos: {best_model_name}")
print(f"Holdout temporal: {best_holdout[['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']].round(4).to_dict()}")
print(f"GroupKFold de activos: {best_group[['precision', 'recall', 'f1', 'roc_auc', 'pr_auc']].round(4).to_dict()}")
print("Matriz de confusión:")
print(cm)


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Mejor modelo por GroupKFold de activos: LogisticRegression
Holdout temporal: {'precision': 0.4060734463276836, 'recall': 0.6748826291079812, 'f1': 0.5070546737213404, 'roc_auc': 0.6784797081683147, 'pr_auc': 0.46080806722030465}
GroupKFold de activos: {'precision': 0.3580453347216387, 'recall': 0.6191691511477874, 'f1': 0.45354917604240697, 'roc_auc': 0.6664586722917238, 'pr_auc': 0.39687788571940763}
Matriz de confusión:
[[1307  841]
 [ 277  575]]


## 5. Entrenamiento final, importancia de variables y artefactos

In [6]:

final_pipeline = build_pipeline(best_model_name)
final_pipeline.fit(X, y)

model_bundle = {
    "pipeline": final_pipeline,
    "features": FEATURES,
    "categorical_features": CATEGORICAL,
    "numeric_features": NUMERIC,
    "target": TARGET,
    "model_name": best_model_name,
    "threshold": 0.5,
}
joblib.dump(model_bundle, MODEL_DIR / "failure_risk_30d.joblib")

feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
final_model = final_pipeline.named_steps["model"]
importance = None

if hasattr(final_model, "feature_importances_"):
    importance = np.asarray(final_model.feature_importances_, dtype=float)
elif hasattr(final_model, "coef_"):
    importance = np.abs(np.asarray(final_model.coef_).ravel())

if importance is not None and len(importance) == len(feature_names):
    importance_df = (
        pd.DataFrame({"feature": feature_names, "importance": importance})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    importance_df.to_csv(REPORTS / "feature_importance.csv", index=False)

    top = importance_df.head(20).sort_values("importance")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top["feature"], top["importance"], color="#2563eb")
    ax.set_title(f"Importancia de variables: {best_model_name}")
    ax.set_xlabel("Importancia")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(importance_df.head(15).to_string(index=False))
else:
    print(f"{best_model_name} no expone importancias de variables directamente.")

metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": str(DATASET.relative_to(ROOT)),
    "rows": int(len(df)),
    "positive_rate": float(y.mean()),
    "best_model": best_model_name,
    "selection_strategy": "group_kfold_asset",
    "threshold": 0.5,
    "holdout_metrics": {
        key: float(best_holdout[key])
        for key in ["precision", "recall", "f1", "roc_auc", "pr_auc"]
    },
    "group_kfold_metrics": {
        key: float(best_group[key])
        for key in ["precision", "recall", "f1", "roc_auc", "pr_auc"]
    },
    "features": FEATURES,
    "model_path": "models/failure_risk_30d.joblib",
}

with (REPORTS / "model_metadata.json").open("w", encoding="utf-8") as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2)

print(f"Modelo guardado en: {MODEL_DIR / 'failure_risk_30d.joblib'}")
print(json.dumps(metadata, ensure_ascii=False, indent=2))


/usr/local/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


                                  feature  importance
              categorical__criticidad_low    0.552020
         categorical__criticidad_critical    0.516920
categorical__tipo_activo_Bomba industrial    0.454858
       categorical__sede_Planta principal    0.417040
    categorical__sede_Centro de servicios    0.394778
       categorical__sede_Centro logístico    0.375542
      categorical__tipo_activo_Envasadora    0.371915
     categorical__tipo_activo_Montacargas    0.363988
  categorical__sede_Patio de contenedores    0.362841
   categorical__sede_Planta de confección    0.359778
categorical__tipo_activo_Cortadora textil    0.349930
           categorical__tipo_activo_Telar    0.348034
       categorical__sede_Planta de tejido    0.265797
                 numeric__antiguedad_dias    0.262825
       categorical__tipo_activo_Compresor    0.261209
Modelo guardado en: /workspace/models/failure_risk_30d.joblib
{
  "generated_at": "2026-11-03T21:26:30.490791+00:00",
  "dataset": "data

In [7]:

preprocessor_fitted = final_pipeline.named_steps["preprocessor"]
numeric_fitted = preprocessor_fitted.named_transformers_["numeric"]
categorical_fitted = preprocessor_fitted.named_transformers_["categorical"]

numeric_imputer = numeric_fitted.named_steps["imputer"]
numeric_scaler = numeric_fitted.named_steps["scaler"]
categorical_imputer = categorical_fitted.named_steps["imputer"]
onehot = categorical_fitted.named_steps["onehot"]

if not hasattr(final_model, "coef_"):
    raise TypeError("La exportación JSON solo admite modelos lineales con coef_.")

portable_model = {
    "version": 1,
    "model_name": best_model_name,
    "model_type": "logistic_regression",
    "target": TARGET,
    "threshold": 0.5,
    "numeric_features": NUMERIC,
    "categorical_features": CATEGORICAL,
    "numeric_impute": {
        feature: float(value)
        for feature, value in zip(NUMERIC, numeric_imputer.statistics_)
    },
    "numeric_mean": {
        feature: float(value)
        for feature, value in zip(NUMERIC, numeric_scaler.mean_)
    },
    "numeric_scale": {
        feature: float(value)
        for feature, value in zip(NUMERIC, numeric_scaler.scale_)
    },
    "categorical_impute": {
        feature: str(value)
        for feature, value in zip(CATEGORICAL, categorical_imputer.statistics_)
    },
    "categories": {
        feature: [str(category) for category in categories]
        for feature, categories in zip(CATEGORICAL, onehot.categories_)
    },
    "feature_names": [str(name) for name in feature_names],
    "coefficients": [float(value) for value in np.asarray(final_model.coef_).ravel()],
    "intercept": float(np.asarray(final_model.intercept_).ravel()[0]),
}

with (MODEL_DIR / "failure_risk_30d.json").open("w", encoding="utf-8") as handle:
    json.dump(portable_model, handle, ensure_ascii=False, indent=2)

print(f"Modelo portable guardado en: {MODEL_DIR / 'failure_risk_30d.json'}")


Modelo portable guardado en: /workspace/models/failure_risk_30d.json


Artefactos generados:

- `models/failure_risk_30d.joblib`
- `models/failure_risk_30d.json`
- `reports/model_comparison.csv` y `reports/model_comparison_folds.csv`
- `reports/model_metadata.json`
- `reports/feature_importance.csv`
- `reports/figures/model_comparison_pr_auc.png`, `confusion_matrix_best_model.png` y `feature_importance.png`

El umbral queda en 0.5; para operación conviene ajustarlo según la capacidad del equipo de mantenimiento y el costo de una falla no atendida.